In [1]:
import geopandas as gpd
from pathlib import Path

# Carga el shapefile de pluviómetros SAIH (descargado manualmente del MITECO)
ruta_shp = Path("../data/raw/saih_inventario/capassaih-pluviometria/Pluviometria.shp")
pluv = gpd.read_file(ruta_shp)

print(f"Total pluviómetros SAIH en España: {len(pluv)}")
print(f"Sistema de coordenadas: {pluv.crs}")
print(f"\nColumnas: {list(pluv.columns)}")
pluv.head()

Total pluviómetros SAIH en España: 1353
Sistema de coordenadas: EPSG:4258

Columnas: ['OBJECTID', 'IDPUNTO', 'DESCRIPCIO', 'IDSAIH', 'TIPO', 'geometry']


,OBJECTID,IDPUNTO,DESCRIPCIO,IDSAIH,TIPO,geometry
0,963,I07_SALADO_MENG,I07_SALADO_MENG,GUADALQUIVIR,Pluviometría (mm),POINT (-3.8075 37.98144)
1,978,AR49 AZUD DEL ENCIN,AR49 AZUD DEL ENCIN,TAJO,Pluviometría (mm),POINT (-6.42594 39.94885)
2,994,AR22 COFIO EN SAN MARTIN,AR22 COFIO EN SAN MARTIN,TAJO,Pluviometría (mm),POINT (-4.32259 40.41204)
3,1011,E_11 BELENA,E_11 BELENA,TAJO,Pluviometría (mm),POINT (-3.19433 40.93034)
4,1033,EM3-02,CUMBRE MAYORES,GUADIANA,Pluviometría (mm),POINT (-6.64619 38.05858)


In [3]:
# Reproyectar a WGS84 (lat/lon) si está en otra proyección
if pluv.crs and pluv.crs.to_epsg() != 4326:
    pluv = pluv.to_crs(epsg=4326)

# Inspecciona qué valores únicos hay en columnas categóricas
for col in pluv.columns:
    if col != "geometry" and pluv[col].dtype == "object":
        unicos = pluv[col].nunique()
        if unicos < 20:  # solo columnas con pocas categorías (posibles filtros)
            print(f"\n{col} ({unicos} únicos):")
            print(pluv[col].value_counts())

In [4]:
print(pluv["IDSAIH"].unique())
print(f"\nTotal demarcaciones: {pluv['IDSAIH'].nunique()}")
print(f"\nPluviómetros por demarcación:")
print(pluv["IDSAIH"].value_counts())

<ArrowStringArray>
[       'GUADALQUIVIR',                'TAJO',            'GUADIANA',
                'EBRO',                 'ACA',              'NORTEI',
               'JUCAR',              'SEGURA', 'AGENCIAANDALUZAAGUA',
               'DUERO']
Length: 10, dtype: str

Total demarcaciones: 10

Pluviómetros por demarcación:
IDSAIH
EBRO                   356
GUADALQUIVIR           198
TAJO                   191
GUADIANA               186
JUCAR                  124
AGENCIAANDALUZAAGUA    103
NORTEI                  81
SEGURA                  65
ACA                     48
DUERO                    1
Name: count, dtype: int64


In [7]:
# Filtra al Júcar (ajusta el valor si la celda anterior te dice otra escritura)
saih_jucar = pluv[pluv["IDSAIH"] == "JUCAR"].copy()

# Extrae lat y lon de la geometría
saih_jucar["lon"] = saih_jucar.geometry.x
saih_jucar["lat"] = saih_jucar.geometry.y

print(f"Pluviómetros SAIH-CHJ: {len(saih_jucar)}")
saih_jucar[["IDPUNTO", "DESCRIPCIO", "lat", "lon"]].head(10)

Pluviómetros SAIH-CHJ: 124


,IDPUNTO,DESCRIPCIO,lat,lon
53,PLUVIO VALLIBONA,VALLIBONA,40.584792,0.001111
54,PLUVIO VALBONA,VALBONA,40.254068,-0.818062
55,PLUVIO PINA,PINA,40.029234,-0.628123
56,PLUVIO ALBARRACIN,ALBARRACIN,40.431020,-1.436050
57,PLUVIO LANDETE,LANDETE,39.911618,-1.371545
58,PLUVIO BENAGEBER,BENAGEBER,39.727324,-1.099924
59,PLUVIO ZAGRA,ZAGRA,39.843047,-1.144831
86,PLUVIO LA CUEVARRUZ,LA CUEVARRUZ,39.936402,-0.984184
87,PLUVIO FORATA,FORATA,39.341183,-0.864723
88,PLUVIO COFRENTES,COFRENTES,39.263031,-1.096256


In [8]:
import json
import pandas as pd
import sys
sys.path.insert(0, "..")
from src.ingest.aemet import parse_coord

with open("../data/raw/aemet/inventario_estaciones.json", encoding="utf-8") as f:
    inv = pd.DataFrame(json.load(f))

provincias_jucar = ["VALENCIA", "CUENCA", "ALBACETE", "CASTELLON", "TERUEL"]
aemet_jucar = inv[inv["provincia"].isin(provincias_jucar)].copy()
aemet_jucar["lat"] = aemet_jucar["latitud"].apply(parse_coord)
aemet_jucar["lon"] = aemet_jucar["longitud"].apply(parse_coord)

print(f"Estaciones AEMET en la zona: {len(aemet_jucar)}")
print(f"Pluviómetros SAIH-CHJ: {len(saih_jucar)}")
print(f"Total combinado: {len(aemet_jucar) + len(saih_jucar)} puntos")

Estaciones AEMET en la zona: 77
Pluviómetros SAIH-CHJ: 124
Total combinado: 201 puntos


In [10]:
import folium
from pathlib import Path

# Estaciones AEMET que descargaste durante el TFM
descargadas_aemet = {"8414A", "8293X", "8175", "8416Y", "8500A", "8025", "8203O", "8058X"}

m = folium.Map(location=[39.5, -1.2], zoom_start=8, tiles="CartoDB positron")

# --- AEMET en rojo ---
for _, row in aemet_jucar.iterrows():
    if pd.isna(row["lat"]) or pd.isna(row["lon"]):
        continue
    es_descargada = row["indicativo"] in descargadas_aemet
    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=8 if es_descargada else 4,
        color="darkred",
        fill=True,
        fill_color="red",
        fill_opacity=0.8 if es_descargada else 0.5,
        weight=2 if es_descargada else 1,
        popup=f"<b>AEMET — {row['nombre']}</b><br>Código: {row['indicativo']}<br>Provincia: {row['provincia']}",
    ).add_to(m)

# --- SAIH-CHJ en azul ---
for _, row in saih_jucar.iterrows():
    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=3,
        color="navy",
        fill=True,
        fill_color="blue",
        fill_opacity=0.7,
        weight=1,
        popup=f"<b>SAIH — {row['DESCRIPCIO']}</b><br>ID: {row['IDPUNTO']}",
    ).add_to(m)

# Leyenda fija en la esquina inferior-izquierda
legend_html = """
<div style="
    position: fixed;
    bottom: 30px;
    left: 30px;
    width: 220px;
    background: white;
    border: 2px solid #555;
    border-radius: 6px;
    padding: 12px;
    font-size: 13px;
    font-family: Arial, sans-serif;
    z-index: 9999;
    box-shadow: 2px 2px 6px rgba(0,0,0,0.3);
    line-height: 1.6;
">
<b style="font-size:14px;">Red de observación</b><br>
<span style="display:inline-block; width:14px; text-align:center;">
  <svg width="14" height="14" style="vertical-align:middle">
    <circle cx="7" cy="7" r="6" fill="red" fill-opacity="0.85"
            stroke="darkred" stroke-width="1.5"/>
  </svg>
</span>
AEMET (descargada)<br>
<span style="display:inline-block; width:14px; text-align:center;">
  <svg width="14" height="14" style="vertical-align:middle">
    <circle cx="7" cy="7" r="4" fill="red" fill-opacity="0.5"
            stroke="darkred" stroke-width="1"/>
  </svg>
</span>
AEMET (disponible)<br>
<span style="display:inline-block; width:14px; text-align:center;">
  <svg width="14" height="14" style="vertical-align:middle">
    <circle cx="7" cy="7" r="3" fill="blue" fill-opacity="0.7"
            stroke="navy" stroke-width="1"/>
  </svg>
</span>
SAIH-CHJ
</div>
"""

m.get_root().html.add_child(folium.Element(legend_html))

# Guarda HTML
out_path = Path("../data/processed/mapa_aemet_saih.html")
out_path.parent.mkdir(parents=True, exist_ok=True)
m.save(str(out_path))
print(f"Mapa guardado en: {out_path.resolve()}")

m

Mapa guardado en: C:\Users\Maria\Documents\tfm-flood-warning\data\processed\mapa_aemet_saih.html
